## Testing automation for LLM apps

When developing applications using generative AI, the behavior of models is less predictable compared to traditional software.   
This unpredictability makes systematic testing even more crucial in saving development time and costs.   
Continuous integration, an essential aspect of LLMOps, involves making small, incremental changes to the software and rigorously testing them to identify issues early when they are simpler to address.   
By implementing a strong automated testing pipeline, you can detect and resolve bugs before they escalate, making them easier and less expensive to fix. Automated testing enables your team to concentrate on creating new features, allowing for faster iteration and product releases.  

In this code sample we are going to present the azure.ai.evaluations library that can be used for automated testing.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
api_version = "2024-02-15-preview"

AZURE_SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
AZURE_AISTUDIO_PROJECT_RESOURCE_GROUP = os.getenv("AZURE_AISTUDIO_PROJECT_RESOURCE_GROUP")
AZURE_AISTUDIO_PROJECT_NAME = os.getenv("AZURE_AISTUDIO_PROJECT_NAME")


In [2]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    ContentSafetyEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
    ViolenceEvaluator,
    SexualEvaluator,
    SelfHarmEvaluator,
    HateUnfairnessEvaluator,
)

from azure.identity import DefaultAzureCredential

try:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    print(ex)
    

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
)

In [3]:
groundedness_evaluator = GroundednessEvaluator(model_config=model_config)
groundedness_evaluator(
    response="Paris is the capital of France.",
    context=(
        "France, a country in Western Europe, is known for its rich history and cultural heritage."
        "The city of Paris, located in the northern part of the country, serves as its capital."
        "Paris is renowned for its art, fashion, and landmarks such as the Eiffel Tower and the Louvre Museum."
    ),
)

{'gpt_groundedness': 5.0}

In [4]:
coherence_evaluator = CoherenceEvaluator(model_config=model_config)
coherence_evaluator(
    query="What is the capital of France?", 
    response="Paris is the capital of France."
)

{'gpt_coherence': 5.0}

In [5]:
relevance_eval = RelevanceEvaluator(model_config=model_config)
relevance_eval(
    query="What is the capital of France?", 
    response="Paris is the capital of France.",
    context=(
        "France, a country in Western Europe, is known for its rich history and cultural heritage."
        "The city of Paris, located in the northern part of the country, serves as its capital."
        "Paris is renowned for its art, fashion, and landmarks such as the Eiffel Tower and the Louvre Museum."
    ),
)

{'gpt_relevance': 5.0}

In [6]:
# Similarity
similarity_eval = SimilarityEvaluator(model_config)
score = similarity_eval(
    query="What is the capital of France?", 
    response="Paris is the capital of France.",
    ground_truth="Tokyo is Japan's capital.",
)
print(score)

{'gpt_similarity': 1.0}
